In [1]:
from pyro_cases.utils.run_amortized_favi_with_fixed_design import train_and_test_amortized_favi_with_fixed_design
from pyro_cases.utils.vae_dict import vae_dict
import torch
from pathlib import Path
import os
from pyro_cases.utils.base_vae import BaseVAEwRegister

In [2]:
device = torch.device("cuda:6")

In [3]:
def move_dict_to_cpu(pre_dict: dict):
    return {
        k: v.to(device="cpu") if isinstance(v, torch.Tensor) else v
        for k, v in pre_dict.items()
    }

In [4]:
test_sample_dicts = {}
for i, task_name in enumerate(vae_dict.keys()):
    print(f"[{i}] run task {task_name}")
    out_dict = train_and_test_amortized_favi_with_fixed_design(task_name=task_name,
                                                                seed=123456,
                                                                device=device,
                                                                lr=1e-3,
                                                                lr_schedule="cosine_annealing",
                                                                batch_size=1024,
                                                                network_width=256,
                                                                steps=5,
                                                                test_seed=7272,
                                                                num_test_obs=1000,
                                                                show_progress=False,
                                                                silent=False,
                                                                return_vae=True,
                                                                suppress_error=False,
                                                                nn_type="deep_set")
    test_sample_dicts[task_name] = move_dict_to_cpu(out_dict["favi_test_sample_dict"])

[0] run task gaussian_linear
[task gaussian_linear seed 123456 completes] task start time: Tue Aug  5 10:49:09 2025; task end time: Tue Aug  5 10:49:10 2025; cost time: 0.0 seconds
[1] run task gaussian_linear_uniform
[task gaussian_linear_uniform seed 123456 completes] task start time: Tue Aug  5 10:49:10 2025; task end time: Tue Aug  5 10:49:10 2025; cost time: 0.0 seconds
[2] run task slcp
[task slcp seed 123456 completes] task start time: Tue Aug  5 10:49:10 2025; task end time: Tue Aug  5 10:49:10 2025; cost time: 0.0 seconds
[3] run task slcp_distractors
[task slcp_distractors seed 123456 completes] task start time: Tue Aug  5 10:49:10 2025; task end time: Tue Aug  5 10:49:10 2025; cost time: 0.1 seconds
[4] run task bernoulli_glm_raw
[task bernoulli_glm_raw seed 123456 completes] task start time: Tue Aug  5 10:49:10 2025; task end time: Tue Aug  5 10:49:10 2025; cost time: 0.1 seconds
[5] run task bernoulli_glm
[task bernoulli_glm seed 123456 completes] task start time: Tue Aug 

In [5]:
extract_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_08-02_deep_set_favi_with_fixed_design_output")
favi_or_elbo = "favi"
output_file_path = Path("/data/scratch/pduan/new_gcvi_output/gcvi_08-02_deep_set_favi_with_fixed_design_test_summary.pt")

In [6]:
extract_files = os.listdir(extract_result_dir)

In [7]:
print(f"# files: {len(extract_files)}")

# files: 117


In [8]:
valid_files = []
for cf in extract_files:
    extract_results = torch.load(extract_result_dir / cf, map_location="cpu")
    find_error = any([r[f"{favi_or_elbo}_error"] is not None for r in extract_results])
    if find_error:
        continue
    valid_files.append(cf)

In [9]:
len(valid_files)

117

In [12]:
for i, vf in enumerate(valid_files):
    print(f"[{i + 1}] {vf}")
    extract_results = torch.load(extract_result_dir / vf, map_location="cpu")

    task_name = extract_results[0]["task"]
    cur_vae: BaseVAEwRegister = vae_dict[task_name](hidden_dim=1, use_neural_network=False)
    cur_vae.do_register(1)
    
    g_test_sample_dict = test_sample_dicts[task_name]
    g_test_x = cur_vae.extract_x_as_set(batch_size=1000, sample_dict=g_test_sample_dict)
    g_test_theta = cur_vae.extract_theta(g_test_sample_dict)
    ori_test_x = extract_results[0]["favi_test_result_dict"]["obs"]
    ori_test_theta = extract_results[0]["favi_test_result_dict"]["true_theta"]
    assert torch.allclose(ori_test_x, g_test_x)
    assert torch.allclose(ori_test_theta, g_test_theta)

[1] pyro_t_arm_kidiq_interaction_c_mn_96.pt
[2] pyro_t_arm_electric_multi_preds_mn_96.pt
[3] pyro_t_arm_wells_probit_mn_96.pt
[4] pyro_t_arm_kidiq_interaction_mn_96.pt
[5] pyro_t_arm_electric_1b_chr_mn_96.pt
[6] pyro_t_arm_radon_no_pool_mn_96.pt
[7] pyro_t_arm_radon_group_chr_mn_96.pt
[8] pyro_t_arm_sesame_one_pred_a_mn_96.pt
[9] pyro_t_arm_latent_glm_mn_96.pt
[10] pyro_t_arm_wells_mn_96.pt
[11] pyro_t_arm_anova_randon_nopred_mn_96.pt
[12] pyro_t_arm_congress_mn_96.pt
[13] pyro_t_arm_y_x_mn_96.pt
[14] pyro_t_arm_kidiq_multi_preds_mn_96.pt
[15] pyro_t_arm_mesquite_volume_mn_96.pt
[16] pyro_t_gaussian_mixture_mn_96.pt
[17] pyro_t_arm_radon_vary_intercept_floor_chr_mn_96.pt
[18] pyro_t_arm_radon_no_pool_chr_mn_96.pt
[19] pyro_t_arm_sesame_one_pred_2b_mn_96.pt
[20] pyro_t_arm_wells_daae_c_mn_96.pt
[21] pyro_t_arm_radon_intercept_chr_mn_96.pt
[22] pyro_t_arm_hiv_chr_mn_96.pt
[23] pyro_t_arm_electric_1a_mn_96.pt


AssertionError: 